In [1]:
try:
    import google.colab  # type: ignore
    from google.colab import output, drive
    # Authorize colab to access google drive
    drive.mount('/content/drive')
    embeddings_path ='/content/drive/My Drive/CIS 5200 Final Project/Data/appliances_reviews_012023_062023 v1.4.0.ftr'
    
    COLAB = True
    !pip -q install pandas pyarrow fastparquet numpy matplotlib
    
except:
    COLAB = False
    from IPython import get_ipython  # type: ignore

    ipython = get_ipython()
    assert ipython is not None
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")
    
    embeddings_path ='../data/appliances_reviews_012023_062023 v1.4.0.ftr'
finally:
    import gc, os, math, json
    from pathlib import Path
    import numpy as np
    import pandas as pd
    path = Path(embeddings_path)
    assert path.exists(), f"File not found: {path}"

In [2]:
df = pd.read_feather(path)
print("Original shape:", df.shape)
cols = list(df.columns)
print("Columns:", len(cols))
print(cols)
# del df_head; gc.collect()

Original shape: (105998, 1299)
Columns: 1299
['review_id', 'rating', 'title', 'text', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'text_len', 'text_words', 'inter_review_time', 'main_category', 'average_rating', 'rating_number', 'price', 'store', 'categories', 'MPN_1', 'MPN_2', 'MPN_3', 'MPN_4', 'MPN_5', 'MPN_6', 'MPN_7', 'MPN_8', 'MPN_9', 'MPN_10', 'MPN_11', 'MPN_12', 'MPN_13', 'MPN_14', 'MPN_15', 'MPN_16', 'MPN_17', 'MPN_18', 'MPN_19', 'MPN_20', 'MPN_21', 'MPN_22', 'MPN_23', 'MPN_24', 'MPN_25', 'MPN_26', 'MPN_27', 'MPN_28', 'MPN_29', 'MPN_30', 'MPN_31', 'MPN_32', 'MPN_33', 'MPN_34', 'MPN_35', 'MPN_36', 'MPN_37', 'MPN_38', 'MPN_39', 'MPN_40', 'MPN_41', 'MPN_42', 'MPN_43', 'MPN_44', 'MPN_45', 'MPN_46', 'MPN_47', 'MPN_48', 'MPN_49', 'MPN_50', 'MPN_51', 'MPN_52', 'MPN_53', 'MPN_54', 'MPN_55', 'MPN_56', 'MPN_57', 'MPN_58', 'MPN_59', 'MPN_60', 'MPN_61', 'MPN_62', 'MPN_63', 'MPN_64', 'MPN_65', 'MPN_66', 'MPN_67', 'MPN_68', 'MPN_69', 'MPN_70', 'MPN_71'

In [3]:
df.head(1)

,review_id,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,...,ResNet503,ResNet504,ResNet505,ResNet506,ResNet507,ResNet508,ResNet509,ResNet510,ResNet511,ResNet512
0,1,3.0,Needs hose clamps,Needs metal hose clamps not plastic ties.... C...,B00004YWK2,B00004YWK2,AFFPAJDCW7NSKE4FZWBRWETUKZ2A,2023-02-18 02:32:06.278,0,True,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# **Stage 0: Data Prep**

In [4]:
# Drop raw text columns (already captured by embeddings) & old inter_review_time (we'll recompute)
drop_cols = [c for c in ['index', 'title', 'text', 'inter_review_time'] if c in df.columns]
print("Dropping columns:", drop_cols)
df = df.drop(columns=drop_cols)
print("Shape after drop:", df.shape)

cols = list(df.columns)
print("Total columns now:", len(cols))
print("Cols:", cols)

Dropping columns: ['title', 'text', 'inter_review_time']
Shape after drop: (105998, 1296)
Total columns now: 1296
Cols: ['review_id', 'rating', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'text_len', 'text_words', 'main_category', 'average_rating', 'rating_number', 'price', 'store', 'categories', 'MPN_1', 'MPN_2', 'MPN_3', 'MPN_4', 'MPN_5', 'MPN_6', 'MPN_7', 'MPN_8', 'MPN_9', 'MPN_10', 'MPN_11', 'MPN_12', 'MPN_13', 'MPN_14', 'MPN_15', 'MPN_16', 'MPN_17', 'MPN_18', 'MPN_19', 'MPN_20', 'MPN_21', 'MPN_22', 'MPN_23', 'MPN_24', 'MPN_25', 'MPN_26', 'MPN_27', 'MPN_28', 'MPN_29', 'MPN_30', 'MPN_31', 'MPN_32', 'MPN_33', 'MPN_34', 'MPN_35', 'MPN_36', 'MPN_37', 'MPN_38', 'MPN_39', 'MPN_40', 'MPN_41', 'MPN_42', 'MPN_43', 'MPN_44', 'MPN_45', 'MPN_46', 'MPN_47', 'MPN_48', 'MPN_49', 'MPN_50', 'MPN_51', 'MPN_52', 'MPN_53', 'MPN_54', 'MPN_55', 'MPN_56', 'MPN_57', 'MPN_58', 'MPN_59', 'MPN_60', 'MPN_61', 'MPN_62', 'MPN_63', 'MPN_64', 'MPN_65', 'MPN_66', 'MPN_67', '

In [5]:
# Identify/Define Column Groups we're working with
# Remove "index (redundant and unecessary), title and text (encoded into MPNet), inter_review_time (we're computing this)"
ID_COLS   = [c for c in ['review_id', 'asin', 'parent_asin', 'user_id'] if c in cols]
TIME_COL  = 'timestamp'
META_COLS = [c for c in ['rating', 'helpful_vote',
                         'verified_purchase', 'text_len',
                         'text_words','main_category',
                         'average_rating', 'rating_number',
                         'price', 'store', 'categories'] if c in cols]

#split metadata into fields that are specific to review, and those that are shared with other reviews for the same item
ITEM_COLS = [c for c in ['main_category','price', 
                         'store', 'categories'] if c in cols]

REVIEW_COLS = [c for c in ['rating', 'helpful_vote',
                           'verified_purchase', 'text_len', 'text_words'
                           'average_rating', 'rating_number'] if c in cols]

TEXT_PREFIX = 'MPN_'      # text embeddings
IMG_PREFIX  = 'ResNet'    # image embeddings

TEXT_COLS = [c for c in cols if c.startswith(TEXT_PREFIX)]
IMG_COLS  = [c for c in cols if c.startswith(IMG_PREFIX)]

print(f"\nText embedding dims:  {len(TEXT_COLS)}")
print(f"Image embedding dims: {len(IMG_COLS)}")



Text embedding dims:  768
Image embedding dims: 512


In [6]:
# Parse Timestamp
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors='coerce') #replace missing values with NaN

# Product key (unique item id)
PRODUCT_COL = 'parent_asin' if 'parent_asin' in df.columns else 'asin'

# has_image flag from ResNet block
if IMG_COLS:
    img_block = df[IMG_COLS].to_numpy(copy=False)
    df['has_image'] = (np.abs(img_block).sum(axis=1) > 0).astype('int8')
else:
    df['has_image'] = 0
print("has_image rate:", float(df['has_image'].mean()))

# Compute inter-review time per product
df = df.sort_values([PRODUCT_COL, TIME_COL])  #within each product, reviews odered chronologically by timestamp
df['next_time'] = df.groupby(PRODUCT_COL)[TIME_COL].shift(-1)
delta = (df['next_time'] - df[TIME_COL])


has_image rate: 0.08162418158833186


In [7]:
# Working on the inter-review time

# In hours (can be changed)
df['y_hours']   = delta.dt.total_seconds() / 3600.0
df['censored']  = df['y_hours'].isna().astype('int8')

# Drop last review per product (censored) for supervised regression
df_sup = df[df['censored'] == 0].copy()
df_sup['y_log'] = np.log1p(df_sup['y_hours'])

print("\nSupervised rows (non-censored):", len(df_sup), "of", len(df))

# Calendar features (possibly used later interesting to have)
df_sup['dow']   = df_sup[TIME_COL].dt.dayofweek.astype('int8')  # 0=Mon
df_sup['month'] = df_sup[TIME_COL].dt.month.astype('int8')
df_sup['hour']  = df_sup[TIME_COL].dt.hour.astype('int8')

df_sup['rev_index'] = df_sup.groupby(PRODUCT_COL).cumcount().astype('int32') + 1


Supervised rows (non-censored): 94297 of 105998


In [8]:
df.head(20) #Properly Ordered

,review_id,rating,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,text_len,text_words,...,ResNet507,ResNet508,ResNet509,ResNet510,ResNet511,ResNet512,has_image,next_time,y_hours,censored
5,6,5.0,B00004YWK2,B00004YWK2,AFNA7RNBEH66UUMHPEE7XJFB3MYA,2023-01-03 12:37:29.824,1,True,175,33,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,2023-01-03 16:32:51.697,3.922743,0
4,5,5.0,B00004YWK2,B00004YWK2,AHKTIX6L7FKPYDENUALEPNPMAIHQ,2023-01-03 16:32:51.697,0,True,708,148,...,0.468356,0.669414,0.217397,1.138075,0.267219,0.775042,1,2023-01-29 19:37:40.587,627.080247,0
3,4,5.0,B00004YWK2,B00004YWK2,AGO4SBTXOUTKYMHKQQNX7ZFDQSFA,2023-01-29 19:37:40.587,0,True,115,24,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,2023-02-07 14:55:58.766,211.305050,0
2,3,1.0,B00004YWK2,B00004YWK2,AGA5X6NUWSQM42KDFJLO25JBXOEA,2023-02-07 14:55:58.766,0,True,47,9,...,1.773487,0.983913,0.220921,1.707405,0.076411,0.557772,1,2023-02-10 00:31:39.499,57.594648,0
1,2,1.0,B00004YWK2,B00004YWK2,AEI6B25VF65CG2HPBQ2FNBG7IQKA,2023-02-10 00:31:39.499,0,True,47,9,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,2023-02-18 02:32:06.278,194.007439,0
0,1,3.0,B00004YWK2,B00004YWK2,AFFPAJDCW7NSKE4FZWBRWETUKZ2A,2023-02-18 02:32:06.278,0,True,108,18,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,NaT,NaN,1
7,8,1.0,B00005QTXI,B00005QTXI,AGDIQN3NMLCFTPJQQ3JUPM24HQZA,2023-01-05 20:42:19.154,0,True,187,38,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,2023-01-30 17:27:55.042,596.759969,0
6,7,5.0,B00005QTXI,B00005QTXI,AEAFYWICODCX7LUMTKXVTGVMITNA,2023-01-30 17:27:55.042,0,True,106,21,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,NaT,NaN,1
15,16,1.0,B00009W3GJ,B00009W3GJ,AHMKLFBWI3V6PCXM2MHFGRFOWP6Q,2023-02-17 23:12:23.917,0,True,70,12,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,2023-02-24 01:15:19.973,146.048904,0
14,15,1.0,B00009W3GJ,B00009W3GJ,AFPWPNSVJMCIIA35ES75UGJXEVCQ,2023-02-24 01:15:19.973,0,True,337,62,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,2023-03-01 23:28:10.529,142.214043,0


In [9]:
delta.head(20)

5     0 days 03:55:21.873000
4    26 days 03:04:48.890000
3     8 days 19:18:18.179000
2     2 days 09:35:40.733000
1     8 days 02:00:26.779000
0                        NaT
7    24 days 20:45:35.888000
6                        NaT
15    6 days 02:02:56.056000
14    5 days 22:12:50.556000
13    5 days 19:50:31.006000
12    0 days 00:34:31.142000
11    6 days 20:58:47.956000
10   18 days 21:30:16.858000
9    68 days 11:55:22.631000
8                        NaT
35    1 days 05:39:40.871000
34    0 days 19:46:42.935000
33    3 days 01:37:00.175000
32    1 days 08:20:03.813000
dtype: timedelta64[ns]

In [10]:
# Time-based folds
TRAIN_END = pd.Timestamp('2023-04-30 23:59:59')
VALID_END = pd.Timestamp('2023-05-31 23:59:59')

def assign_fold(ts):
    if ts <= TRAIN_END: return 'train'
    if ts <= VALID_END: return 'valid'
    return 'test'

df_sup['fold'] = df_sup[TIME_COL].apply(assign_fold)
print("\nFold counts:")
print(df_sup['fold'].value_counts())

# Quick sanity checks
print("\nTarget stats (y_hours):")
print(df_sup['y_hours'].describe(percentiles=[0.5, 0.9, 0.99]))

print("\nVerified purchase proportion (supervised only):")
print(df_sup['verified_purchase'].value_counts(normalize=True))

# Save Stage 0
df_sup.to_parquet('../data/appliances_stage0_1.0.parquet', index=False)


Fold counts:
fold
train    85376
valid     6330
test      2591
Name: count, dtype: int64

Target stats (y_hours):
count    94297.000000
mean       251.827422
std        431.336111
min          0.000000
50%         76.408680
90%        727.837581
99%       2133.002603
max       4278.727927
Name: y_hours, dtype: float64

Verified purchase proportion (supervised only):
verified_purchase
True     0.918269
False    0.081731
Name: proportion, dtype: float64


In [11]:
df_sup.head(20)

,review_id,rating,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,text_len,text_words,...,has_image,next_time,y_hours,censored,y_log,dow,month,hour,rev_index,fold
5,6,5.0,B00004YWK2,B00004YWK2,AFNA7RNBEH66UUMHPEE7XJFB3MYA,2023-01-03 12:37:29.824,1,True,175,33,...,0,2023-01-03 16:32:51.697,3.922743,0,1.593866,1,1,12,1,train
4,5,5.0,B00004YWK2,B00004YWK2,AHKTIX6L7FKPYDENUALEPNPMAIHQ,2023-01-03 16:32:51.697,0,True,708,148,...,1,2023-01-29 19:37:40.587,627.080247,0,6.442668,1,1,16,2,train
3,4,5.0,B00004YWK2,B00004YWK2,AGO4SBTXOUTKYMHKQQNX7ZFDQSFA,2023-01-29 19:37:40.587,0,True,115,24,...,0,2023-02-07 14:55:58.766,211.305050,0,5.358024,6,1,19,3,train
2,3,1.0,B00004YWK2,B00004YWK2,AGA5X6NUWSQM42KDFJLO25JBXOEA,2023-02-07 14:55:58.766,0,True,47,9,...,1,2023-02-10 00:31:39.499,57.594648,0,4.070643,1,2,14,4,train
1,2,1.0,B00004YWK2,B00004YWK2,AEI6B25VF65CG2HPBQ2FNBG7IQKA,2023-02-10 00:31:39.499,0,True,47,9,...,0,2023-02-18 02:32:06.278,194.007439,0,5.273038,4,2,0,5,train
7,8,1.0,B00005QTXI,B00005QTXI,AGDIQN3NMLCFTPJQQ3JUPM24HQZA,2023-01-05 20:42:19.154,0,True,187,38,...,0,2023-01-30 17:27:55.042,596.759969,0,6.393189,3,1,20,1,train
15,16,1.0,B00009W3GJ,B00009W3GJ,AHMKLFBWI3V6PCXM2MHFGRFOWP6Q,2023-02-17 23:12:23.917,0,True,70,12,...,0,2023-02-24 01:15:19.973,146.048904,0,4.990765,4,2,23,1,train
14,15,1.0,B00009W3GJ,B00009W3GJ,AFPWPNSVJMCIIA35ES75UGJXEVCQ,2023-02-24 01:15:19.973,0,True,337,62,...,0,2023-03-01 23:28:10.529,142.214043,0,4.964340,4,2,1,2,train
13,14,5.0,B00009W3GJ,B00009W3GJ,AFIKDA2B76TEOXDKJEHXWLGIIC7Q,2023-03-01 23:28:10.529,0,True,28,5,...,0,2023-03-07 19:18:41.535,139.841946,0,4.947638,2,3,23,3,train
12,13,1.0,B00009W3GJ,B00009W3GJ,AFB4BPNDGY6ELPURIPETVVAP3ZKQ,2023-03-07 19:18:41.535,0,True,207,42,...,0,2023-03-07 19:53:12.677,0.575317,0,0.454457,1,3,19,4,train


## Splitting with random sampling instead of by time

In [16]:
df_sup_sampled = df_sup
df_sup_sampled['fold'] = ''
train_set = df_sup_sampled.sample(frac=0.9,random_state=42)
test_set = df_sup_sampled.drop(train_set.index)
val_set = test_set.sample(frac=0.7,random_state=42)
test_set = test_set.drop(val_set.index)

def assign_fold_sampled(idx):
    if idx in test_set['index'].values:
        return 'test'
    elif idx in val_set['index'].values:
        return 'valid'
    return 'train'

train_set['fold'] = 'train'
val_set['fold'] = 'valid'
test_set['fold'] = 'test'

df_sup_sampled = pd.concat([train_set, val_set, test_set])

# df_sup['fold'] = df_sup['index'].apply(assign_fold_sampled)
print("\nFold counts:")
print(df_sup_sampled['fold'].value_counts())

df_sup_sampled.to_parquet('../data/appliances_stage0_sampled_1.0.parquet', index=False)



Fold counts:
fold
train    84867
valid     6601
test      2829
Name: count, dtype: int64


In [15]:
df_sup_sampled.head(10)

,review_id,rating,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,text_len,text_words,...,has_image,next_time,y_hours,censored,y_log,dow,month,hour,rev_index,fold
28605,28606,1.0,B07WXM6SG5,B07WXM6SG5,AHG2Z4K2M226CVERNHZKPPIVXCEA,2023-02-27 20:02:26.670,0,True,49,8,...,0,2023-03-01 19:06:57.276,47.075168,0,3.872766,0,2,20,9,train
38038,38039,5.0,B0889X1STW,B0889X1STW,AHJENM56CSGXCBHP2O2RQ3QF4DIQ,2023-01-05 19:36:58.140,0,True,11,3,...,0,2023-02-05 18:38:55.664,743.032646,0,6.612085,3,1,19,1,train
54652,54653,4.0,B09D7B1HPS,B09D7B1HPS,AHPQNGIKG445QEO6DCRNI3LIZJ4A,2023-02-07 01:14:56.810,0,True,1606,312,...,0,2023-02-23 07:08:24.736,389.891091,0,5.968429,1,2,1,6,train
59515,59516,5.0,B09NN5SCV6,B09NN5D2NK,AEBEXXCA6FZS3ACJK6WK3HOF7U7Q,2023-03-21 23:32:08.402,0,True,83,14,...,0,2023-03-23 11:03:44.789,35.526774,0,3.598046,1,3,23,18,train
58751,58752,5.0,B00UB441HS,B09MQLYPRR,AEX5UO7SQM6Z2J4NZHE3C3FTWUMQ,2023-03-08 11:41:05.967,0,True,97,21,...,0,2023-03-09 18:02:46.401,30.361232,0,3.445572,2,3,11,64,train
62744,62745,4.0,B09SYS9RQV,B09SYS9RQV,AFX45F7RUZ6IYRYD45ONHFGMGPHA,2023-04-02 16:13:45.244,0,True,84,19,...,0,2023-06-17 03:12:00.620,1810.970938,0,7.502170,6,4,16,1,train
99841,99842,5.0,B0BCQ8Q81F,B0C5H9GKJM,AG4GMN3KPPAJXLPUDSBRQDYLWQCQ,2023-04-08 14:41:34.556,0,True,43,8,...,1,2023-05-24 03:48:19.376,1093.112450,0,6.997699,5,4,14,11,train
60391,60392,5.0,B09Q6FR6FB,B09Q6FR6FB,AEYIMWQU6LBD4VDIDDLUZTNPLVPQ,2023-01-21 20:36:40.973,0,True,16,3,...,0,2023-03-04 06:46:11.134,994.158378,0,6.902902,5,1,20,3,train
81727,81728,5.0,B0BLGMMW79,B0BLGMMW79,AEHW3LFTQPHLS43CW64WXOFRVBMA,2023-01-28 19:01:05.951,0,True,162,33,...,0,2023-03-10 22:28:52.420,987.462908,0,6.896151,5,1,19,1,train
28292,28293,5.0,B01MT0UL8N,B07WTXWC32,AFVYB2QFGGT646UDJIJEUXBCPTCQ,2023-01-28 23:00:06.895,0,True,28,4,...,0,2023-01-29 03:08:23.841,4.138041,0,1.636672,5,1,23,125,train


# **Stage 1: Baselines**

In [17]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [18]:
# 1. Load Stage 0
stage0_path = '../data/appliances_stage0_sampled_1.0.parquet'
df_sup = pd.read_parquet(stage0_path)
print("Loaded df_sup:", df_sup.shape)

# Sanity: check we have y_log and fold
assert 'y_log' in df_sup.columns, "y_log missing"
assert 'fold' in df_sup.columns, "fold missing"


Loaded df_sup: (94297, 1306)


### Baseline A: Naive per-product mean baseline

In [19]:
# Choose product column
PRODUCT_COL = 'parent_asin' if 'parent_asin' in df_sup.columns else 'asin'
print("Using product column:", PRODUCT_COL)

# Split into train/valid/test
train_df = df_sup[df_sup['fold'] == 'train'].copy()
valid_df = df_sup[df_sup['fold'] == 'valid'].copy()
test_df  = df_sup[df_sup['fold'] == 'test'].copy()

print("Train/Valid/Test sizes:", train_df.shape, valid_df.shape, test_df.shape)

# Compute per-product mean on TRAIN ONLY
per_product_mean_hours = train_df.groupby(PRODUCT_COL)['y_hours'].mean()

#Mean for global becaue some products might not appear in train at all and only valid/test
# so for new products we can use generic global mean gap guess
global_mean_hours = train_df['y_hours'].mean()

print("\nGlobal mean inter-review time (hours):", global_mean_hours," (~", global_mean_hours/24, "days)")

Using product column: parent_asin
Train/Valid/Test sizes: (84867, 1306) (6601, 1306) (2829, 1306)

Global mean inter-review time (hours): 251.62280959202175  (~ 10.484283733000906 days)


In [20]:
# predict function for a split
def predict_split_mean_hours(split_df):
    """
    For each row in split_df, predict the product's mean y_hours
    if seen in train; else fall back to global mean.
    """
    y_true_hours = split_df['y_hours'].to_numpy()
    y_true_log = split_df['y_log'].to_numpy()

    # Map product -> mean hours (NaN if unseen)
    y_pred_hours = split_df[PRODUCT_COL].map(per_product_mean_hours).to_numpy()
    # Fill unseen products with global mean
    unseen_mask = np.isnan(y_pred_hours)
    y_pred_hours[unseen_mask] = global_mean_hours

    # Corresponding log predictions
    y_pred_log = np.log1p(y_pred_hours)

    return y_true_hours, y_true_log, y_pred_hours, y_pred_log

# Evaluation function
def evaluate_split(name, y_true_log, y_pred_log, y_true_hours, y_pred_hours):
    mae_log = mean_absolute_error(y_true_log, y_pred_log)
    rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))

    mae_h = mean_absolute_error(y_true_hours, y_pred_hours)
    rmse_h = np.sqrt(mean_squared_error(y_true_hours, y_pred_hours))

    print(f"\n{name} performance (Naive per-product mean):")
    print(f"  MAE_log  = {mae_log:.4f}")
    print(f"  RMSE_log = {rmse_log:.4f}")
    print(f"  MAE_h    = {mae_h:.2f} hours ({mae_h/24:.2f} days)")
    print(f"  RMSE_h   = {rmse_h:.2f} hours ({rmse_h/24:.2f} days)")

# Compute predictions for each split
y_tr_h, y_tr_log, y_tr_pred_h, y_tr_pred_log = predict_split_mean_hours(train_df)
y_va_h, y_va_log, y_va_pred_h, y_va_pred_log = predict_split_mean_hours(valid_df)
y_te_h, y_te_log, y_te_pred_h, y_te_pred_log = predict_split_mean_hours(test_df)

# Evaluate
evaluate_split("TRAIN", y_tr_log, y_tr_pred_log, y_tr_h, y_tr_pred_h)
evaluate_split("VALID", y_va_log, y_va_pred_log, y_va_h, y_va_pred_h)
evaluate_split("TEST",  y_te_log, y_te_pred_log, y_te_h, y_te_pred_h)

# Save predictions
preds_mean = pd.DataFrame({
    'fold': df_sup['fold'],
    'y_log': df_sup['y_log'],
    'y_hours': df_sup['y_hours'],
})

train_idx = df_sup.index[df_sup['fold'] == 'train']
valid_idx = df_sup.index[df_sup['fold'] == 'valid']
test_idx  = df_sup.index[df_sup['fold'] == 'test']

preds_mean.loc[train_idx, 'y_pred_hours_mean'] = y_tr_pred_h
preds_mean.loc[valid_idx, 'y_pred_hours_mean'] = y_va_pred_h
preds_mean.loc[test_idx,  'y_pred_hours_mean'] = y_te_pred_h

"""
preds_mean_path = '/content/drive/My Drive/CIS 5200 Final Project/Data/preds_naive_mean.parquet'
preds_mean.to_parquet(preds_mean_path, index=False)
print("\nSaved naive per-product mean predictions to:", preds_mean_path)
"""


TRAIN performance (Naive per-product mean):
  MAE_log  = 1.0424
  RMSE_log = 1.4675
  MAE_h    = 144.33 hours (6.01 days)
  RMSE_h   = 257.46 hours (10.73 days)

VALID performance (Naive per-product mean):
  MAE_log  = 1.2217
  RMSE_log = 1.6327
  MAE_h    = 223.80 hours (9.32 days)
  RMSE_h   = 444.70 hours (18.53 days)

TEST performance (Naive per-product mean):
  MAE_log  = 1.2154
  RMSE_log = 1.6200
  MAE_h    = 223.73 hours (9.32 days)
  RMSE_h   = 443.75 hours (18.49 days)


'\npreds_mean_path = \'/content/drive/My Drive/CIS 5200 Final Project/Data/preds_naive_mean.parquet\'\npreds_mean.to_parquet(preds_mean_path, index=False)\nprint("\nSaved naive per-product mean predictions to:", preds_mean_path)\n'

### Baseline B: Text-Only Using only MPNet Embeddings, Ridge Regression on **y_log**

In [21]:
# Identify MPNet text embedding columns
TEXT_PREFIX = 'MPN_'
TEXT_COLS = [c for c in df_sup.columns if c.startswith(TEXT_PREFIX)]
print("Num MPNet dims:", len(TEXT_COLS))
print("First 10 text cols:", TEXT_COLS[:10])

# Split into train/valid/test
train_df = df_sup[df_sup['fold'] == 'train'].copy()
valid_df = df_sup[df_sup['fold'] == 'valid'].copy()
test_df  = df_sup[df_sup['fold'] == 'test'].copy()

print("Train/Valid/Test sizes:", train_df.shape, valid_df.shape, test_df.shape)

# Targets (log-space)
y_train_log = train_df['y_log'].to_numpy()
y_valid_log = valid_df['y_log'].to_numpy()
y_test_log  = test_df['y_log'].to_numpy()

# Features (text-only)
X_train_text = train_df[TEXT_COLS].to_numpy(dtype=np.float32)
X_valid_text = valid_df[TEXT_COLS].to_numpy(dtype=np.float32)
X_test_text  = test_df[TEXT_COLS].to_numpy(dtype=np.float32)

# Free DataFrames (save RAM)
del train_df, valid_df, test_df
gc.collect()

# Scale text features (fit on train only)
scaler = StandardScaler(with_mean=True, with_std=True)
X_train_scaled = scaler.fit_transform(X_train_text)  #only firt the scaler on train only
X_valid_scaled = scaler.transform(X_valid_text)
X_test_scaled  = scaler.transform(X_test_text)

# We can free the unscaled arrays if memory is tight
del X_train_text, X_valid_text, X_test_text
gc.collect()

Num MPNet dims: 768
First 10 text cols: ['MPN_1', 'MPN_2', 'MPN_3', 'MPN_4', 'MPN_5', 'MPN_6', 'MPN_7', 'MPN_8', 'MPN_9', 'MPN_10']
Train/Valid/Test sizes: (84867, 1306) (6601, 1306) (2829, 1306)


0

In [22]:
def evaluate_split(name, y_true_log, y_pred_log):
    """
    Print metrics in both log-space and original hours.
    """
    # Log-space metrics
    mae_log  = mean_absolute_error(y_true_log, y_pred_log)
    rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))

    # Convert back to hours
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)

    mae_h  = mean_absolute_error(y_true, y_pred)
    rmse_h = np.sqrt(mean_squared_error(y_true, y_pred))

    print(f"\n{name} performance:")
    print(f"  MAE_log  = {mae_log:.4f}")
    print(f"  RMSE_log = {rmse_log:.4f}")
    print(f"  MAE_h    = {mae_h:.2f} hours ({mae_h/24:.2f} days)")
    print(f"  RMSE_h   = {rmse_h:.2f} hours ({rmse_h/24:.2f} days)")

alphas = [0.1, 0.3, 1.0, 3.0, 10.0, 30.0]
best_alpha = None
best_valid_rmse = float('inf')
best_model = None

for alpha in alphas:
    model = Ridge(alpha=alpha, fit_intercept=True, random_state=0)
    model.fit(X_train_scaled, y_train_log)

    # Predict on valid
    y_valid_pred_log = model.predict(X_valid_scaled)
    rmse_valid = np.sqrt(mean_squared_error(y_valid_log, y_valid_pred_log))

    print(f"alpha={alpha:.3g} -> valid RMSE_log = {rmse_valid:.4f}")

    if rmse_valid < best_valid_rmse:
        best_valid_rmse = rmse_valid
        best_alpha = alpha
        best_model = model

print(f"\nBest alpha by valid RMSE_log: {best_alpha} (RMSE_log={best_valid_rmse:.4f})")

alpha=0.1 -> valid RMSE_log = 1.8095
alpha=0.3 -> valid RMSE_log = 1.8095
alpha=1 -> valid RMSE_log = 1.8095
alpha=3 -> valid RMSE_log = 1.8094
alpha=10 -> valid RMSE_log = 1.8094
alpha=30 -> valid RMSE_log = 1.8093

Best alpha by valid RMSE_log: 30.0 (RMSE_log=1.8093)


In [23]:
# Final evaluation
y_train_pred_log = best_model.predict(X_train_scaled)
y_valid_pred_log = best_model.predict(X_valid_scaled)
y_test_pred_log  = best_model.predict(X_test_scaled)

evaluate_split("TRAIN", y_train_log, y_train_pred_log)
evaluate_split("VALID", y_valid_log, y_valid_pred_log)
evaluate_split("TEST",  y_test_log,  y_test_pred_log)

# Save predictions
preds_df = pd.DataFrame({
    'fold': df_sup['fold'],
    'y_log': df_sup['y_log'],
})

train_idx = df_sup.index[df_sup['fold'] == 'train']
valid_idx = df_sup.index[df_sup['fold'] == 'valid']
test_idx  = df_sup.index[df_sup['fold'] == 'test']

preds_df.loc[train_idx, 'y_pred_log_text_ridge'] = y_train_pred_log
preds_df.loc[valid_idx, 'y_pred_log_text_ridge'] = y_valid_pred_log
preds_df.loc[test_idx,  'y_pred_log_text_ridge'] = y_test_pred_log

"""
preds_path = '/content/drive/My Drive/CIS 5200 Final Project/Data/preds_text_ridge.parquet'
preds_df.to_parquet(preds_path, index=False)
print("\nSaved text-only ridge predictions to:", preds_path)
"""


TRAIN performance:
  MAE_log  = 1.4346
  RMSE_log = 1.7740
  MAE_h    = 220.41 hours (9.18 days)
  RMSE_h   = 457.34 hours (19.06 days)

VALID performance:
  MAE_log  = 1.4636
  RMSE_log = 1.8093
  MAE_h    = 222.28 hours (9.26 days)
  RMSE_h   = 461.63 hours (19.23 days)

TEST performance:
  MAE_log  = 1.4485
  RMSE_log = 1.7911
  MAE_h    = 230.69 hours (9.61 days)
  RMSE_h   = 481.05 hours (20.04 days)


'\npreds_path = \'/content/drive/My Drive/CIS 5200 Final Project/Data/preds_text_ridge.parquet\'\npreds_df.to_parquet(preds_path, index=False)\nprint("\nSaved text-only ridge predictions to:", preds_path)\n'

In [20]:
preds_df.loc[2, 'y_pred_log_text_ridge']  #predicted log gap for the same row i in df_up

4.973424

### Baseline C: metadata only

In [24]:
ITEM_COLS = [c for c in ['main_category','price', 
                         'store', 'categories'] if c in df_sup.columns]

REVIEW_COLS = [c for c in ['rating', 'helpful_vote',
                           'verified_purchase', 'text_len', 'text_words',
                           'average_rating', 'rating_number'] if c in df_sup.columns]

print(f"Num review metadata dims: {len(REVIEW_COLS)}")
print(f"Metadata cols: {REVIEW_COLS}")

train_df = df_sup[df_sup['fold'] == 'train'].copy()
valid_df = df_sup[df_sup['fold'] == 'valid'].copy()
test_df  = df_sup[df_sup['fold'] == 'test'].copy()

# print("Train/Valid/Test sizes:", train_df.shape, valid_df.shape, test_df.shape)

# Targets (log-space)
y_train_log = train_df['y_log'].to_numpy()
y_valid_log = valid_df['y_log'].to_numpy()
y_test_log  = test_df['y_log'].to_numpy()

# Features (text-only)
X_train_metadata = train_df[REVIEW_COLS].to_numpy(dtype=np.float32)
X_valid_metadata = valid_df[REVIEW_COLS].to_numpy(dtype=np.float32)
X_test_metadata  = test_df[REVIEW_COLS].to_numpy(dtype=np.float32)

# Free DataFrames (save RAM)
del train_df, valid_df, test_df
gc.collect()

# Scale text features (fit on train only)
scaler = StandardScaler(with_mean=True, with_std=True)
X_train_scaled = scaler.fit_transform(X_train_metadata)  #only firt the scaler on train only
X_valid_scaled = scaler.transform(X_valid_metadata)
X_test_scaled  = scaler.transform(X_test_metadata)

print("Train/Valid/Test sizes:", X_train_scaled.shape, X_valid_scaled.shape, X_test_scaled.shape)

# We can free the unscaled arrays if memory is tight
del X_train_metadata, X_valid_metadata, X_test_metadata
gc.collect()

Num review metadata dims: 7
Metadata cols: ['rating', 'helpful_vote', 'verified_purchase', 'text_len', 'text_words', 'average_rating', 'rating_number']
Train/Valid/Test sizes: (84867, 7) (6601, 7) (2829, 7)


0

In [25]:
import lightgbm as lgb

lgb_train = lgb.Dataset(X_train_scaled, y_train_log)
lgb_valid = lgb.Dataset(X_valid_scaled, y_valid_log, reference=lgb_train)

params = {
    "boosting_type": "gbdt",
    "objective": "regression",
    "metric": {"l2", "l1", "rmse"},
    "num_leaves": 5,
    "learning_rate": 0.05,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": 1,
}


print("Starting training...")
# train
gbm = lgb.train(
    params, lgb_train, num_boost_round=20, valid_sets=lgb_valid, callbacks=[lgb.early_stopping(stopping_rounds=5)]
)

print("Starting predicting...")
# predict
y_pred = gbm.predict(X_test_scaled, num_iteration=gbm.best_iteration)
# eval
rmse_test = mean_squared_error(y_test_log, y_pred) ** 0.5
print(f"The RMSE of prediction is: {rmse_test}")

Starting training...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001765 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 860
[LightGBM] [Info] Number of data points in the train set: 84867, number of used features: 7
[LightGBM] [Info] Start training from score 4.225528
Training until validation scores don't improve for 5 rounds
Did not meet early stopping. Best iteration is:
[20]	valid_0's rmse: 1.66181	valid_0's l1: 1.33472	valid_0's l2: 2.7616
Starting predicting...
The RMSE of prediction is: 1.6590128215474287


### Baseline D: metadata + text embeddings

In [26]:
ITEM_COLS = [c for c in ['main_category','price', 
                         'store', 'categories'] if c in df_sup.columns]

REVIEW_COLS = [c for c in ['rating', 'helpful_vote',
                           'verified_purchase', 'text_len', 'text_words',
                           'average_rating', 'rating_number'] if c in df_sup.columns]

META_AND_TEXT_COLS = [c for c in df_sup.columns if c.startswith(TEXT_PREFIX) or c in ['rating', 'helpful_vote',
                                                                                      'verified_purchase', 'text_len', 'text_words',
                                                                                      'average_rating', 'rating_number']]

print(f"Num review metadata dims: {len(META_AND_TEXT_COLS)}")
print(f"Metadata cols: {META_AND_TEXT_COLS}")

train_df = df_sup[df_sup['fold'] == 'train'].copy()
valid_df = df_sup[df_sup['fold'] == 'valid'].copy()
test_df  = df_sup[df_sup['fold'] == 'test'].copy()

# print("Train/Valid/Test sizes:", train_df.shape, valid_df.shape, test_df.shape)

# Targets (log-space)
y_train_log = train_df['y_log'].to_numpy()
y_valid_log = valid_df['y_log'].to_numpy()
y_test_log  = test_df['y_log'].to_numpy()

# Features (text-only)
X_train_metadata = train_df[META_AND_TEXT_COLS].to_numpy(dtype=np.float32)
X_valid_metadata = valid_df[META_AND_TEXT_COLS].to_numpy(dtype=np.float32)
X_test_metadata  = test_df[META_AND_TEXT_COLS].to_numpy(dtype=np.float32)

# Free DataFrames (save RAM)
del train_df, valid_df, test_df
gc.collect()

# Scale text features (fit on train only)
scaler = StandardScaler(with_mean=True, with_std=True)
X_train_scaled = scaler.fit_transform(X_train_metadata)  #only firt the scaler on train only
X_valid_scaled = scaler.transform(X_valid_metadata)
X_test_scaled  = scaler.transform(X_test_metadata)

print("Train/Valid/Test sizes:", X_train_scaled.shape, X_valid_scaled.shape, X_test_scaled.shape)

# We can free the unscaled arrays if memory is tight
del X_train_metadata, X_valid_metadata, X_test_metadata

Num review metadata dims: 775
Metadata cols: ['rating', 'helpful_vote', 'verified_purchase', 'text_len', 'text_words', 'average_rating', 'rating_number', 'MPN_1', 'MPN_2', 'MPN_3', 'MPN_4', 'MPN_5', 'MPN_6', 'MPN_7', 'MPN_8', 'MPN_9', 'MPN_10', 'MPN_11', 'MPN_12', 'MPN_13', 'MPN_14', 'MPN_15', 'MPN_16', 'MPN_17', 'MPN_18', 'MPN_19', 'MPN_20', 'MPN_21', 'MPN_22', 'MPN_23', 'MPN_24', 'MPN_25', 'MPN_26', 'MPN_27', 'MPN_28', 'MPN_29', 'MPN_30', 'MPN_31', 'MPN_32', 'MPN_33', 'MPN_34', 'MPN_35', 'MPN_36', 'MPN_37', 'MPN_38', 'MPN_39', 'MPN_40', 'MPN_41', 'MPN_42', 'MPN_43', 'MPN_44', 'MPN_45', 'MPN_46', 'MPN_47', 'MPN_48', 'MPN_49', 'MPN_50', 'MPN_51', 'MPN_52', 'MPN_53', 'MPN_54', 'MPN_55', 'MPN_56', 'MPN_57', 'MPN_58', 'MPN_59', 'MPN_60', 'MPN_61', 'MPN_62', 'MPN_63', 'MPN_64', 'MPN_65', 'MPN_66', 'MPN_67', 'MPN_68', 'MPN_69', 'MPN_70', 'MPN_71', 'MPN_72', 'MPN_73', 'MPN_74', 'MPN_75', 'MPN_76', 'MPN_77', 'MPN_78', 'MPN_79', 'MPN_80', 'MPN_81', 'MPN_82', 'MPN_83', 'MPN_84', 'MPN_85', 'MPN_

In [27]:
import lightgbm as lgb

lgb_train = lgb.Dataset(X_train_scaled, y_train_log)
lgb_valid = lgb.Dataset(X_valid_scaled, y_valid_log, reference=lgb_train)

params = {
    "boosting_type": "gbdt",
    "objective": "regression",
    "metric": {"l2", "l1", "rmse"},
    "num_leaves": 100,
    "learning_rate": 0.05,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": 1,
}


print("Starting training...")
# train
gbm = lgb.train(
    params, lgb_train, num_boost_round=50, valid_sets=lgb_valid, callbacks=[lgb.early_stopping(stopping_rounds=5)]
)

print("Starting predicting...")
# predict
y_pred = gbm.predict(X_test_scaled, num_iteration=gbm.best_iteration)
# eval
rmse_test = mean_squared_error(y_test_log, y_pred) ** 0.5
print(f"The RMSE of prediction is: {rmse_test}")

Starting training...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.100471 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 196700
[LightGBM] [Info] Number of data points in the train set: 84867, number of used features: 775
[LightGBM] [Info] Start training from score 4.225528
Training until validation scores don't improve for 5 rounds
Did not meet early stopping. Best iteration is:
[50]	valid_0's rmse: 1.54885	valid_0's l1: 1.22269	valid_0's l2: 2.39895
Starting predicting...
The RMSE of prediction is: 1.535976512490666
